**Setup toàn bộ môi trường + dataset + code**

In [ ]:
# =========================================================
# CELL 1 - FULL SETUP COLAB
# =========================================================
# ===== ĐI VỀ /content =====
%cd /content
# ===== CHECK GPU =====
import torch

print("=" * 50)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("=" * 50)

# =========================================================
# CLONE GITHUB
# =========================================================

!git clone https://github.com/ThongLuc2k3/Prompt-Guided-XRay-Segmentation.git

# =========================================================
# DOWNLOAD DATASET ZIP TỪ GOOGLE DRIVE
# =========================================================

!gdown --id 1MZIZx7XygG_T_UKiU0H8f_5zWZCgJIUz

# =========================================================
# UNZIP DATASET
# =========================================================

!unzip -q dataset_BTXRD.zip

# =========================================================
# KIỂM TRA DATASET
# =========================================================

print("\nDATASET STRUCTURE:")
!ls dataset_BTXRD

# =========================================================
# COPY DATASET VÀO PROJECT
# =========================================================

!mv dataset_BTXRD Prompt-Guided-XRay-Segmentation/

# =========================================================
# ĐI VÀO PROJECT
# =========================================================

%cd Prompt-Guided-XRay-Segmentation

# =========================================================
# INSTALL REQUIREMENTS
# =========================================================

!pip install -q tqdm opencv-python matplotlib scikit-image gdown

print("\nSETUP DONE!")

**Train**

In [ ]:
# =========================================================
# TRAIN
# =========================================================

!python train.py

**Test + Dice + IoU + BCE**

In [ ]:
# =========================================================
# TEST PHASE (CLEAN - PAPER VERSION)
# =========================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader
import torch.nn as nn

from dataset import BTXRD_Dataset
from models.networks.unet_2D import unet_2D

# =========================================================
# CONFIG
# =========================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "checkpoints/att_unet_best.pth"

# =========================================================
# LOAD MODEL
# =========================================================

model = unet_2D(in_channels=1, n_classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# =========================================================
# DATASET
# =========================================================

test_dataset = BTXRD_Dataset(
    image_dir="dataset_BTXRD/test/images",
    mask_dir="dataset_BTXRD/test/masks",
    img_size=256,
    is_train=False
)

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# =========================================================
# VISUALIZATION CONTROL
# =========================================================
# 👉 đổi index ở đây để xem ảnh bạn muốn
SHOW_INDEX = [0, 5, 10]

# =========================================================
# METRICS
# =========================================================

dice_scores = []
iou_scores = []

bce_loss_fn = nn.BCEWithLogitsLoss()

# =========================================================
# TEST LOOP
# =========================================================

with torch.no_grad():

    for idx, (images, masks) in enumerate(test_loader):

        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        outputs = model(images)

        probs = torch.sigmoid(outputs)

        # -------------------------
        # BINARY PREDICTION
        # -------------------------
        preds = (probs > 0.5).float()

        # -------------------------
        # DICE
        # -------------------------
        intersection = (preds * masks).sum()

        dice = (2.0 * intersection + 1e-5) / (preds.sum() + masks.sum() + 1e-5)

        # -------------------------
        # IOU
        # -------------------------
        union = preds.sum() + masks.sum() - intersection

        iou = (intersection + 1e-5) / (union + 1e-5)

        dice_scores.append(dice.item())
        iou_scores.append(iou.item())

        # =====================================================
        # VISUALIZATION (ONLY SELECTED IMAGES)
        # =====================================================

        if idx in SHOW_INDEX:

            img = images[0, 0].cpu().numpy()
            gt = masks[0, 0].cpu().numpy()
            pred = preds[0, 0].cpu().numpy()
            prob = probs[0, 0].cpu().numpy()

            plt.figure(figsize=(14, 4))

            plt.subplot(1, 4, 1)
            plt.imshow(img, cmap='gray')
            plt.title("Image")

            plt.subplot(1, 4, 2)
            plt.imshow(gt, cmap='gray')
            plt.title("Ground Truth")

            plt.subplot(1, 4, 3)
            plt.imshow(prob, cmap='jet')
            plt.title("Probability")

            plt.subplot(1, 4, 4)
            plt.imshow(pred, cmap='gray')
            plt.title("Prediction")

            plt.tight_layout()
            plt.show()

# =========================================================
# FINAL REPORT (FOR THESIS)
# =========================================================

print("\n" + "=" * 50)
print("📊 FINAL TEST RESULTS")
print("=" * 50)

print(f"Mean Dice : {np.mean(dice_scores):.4f}")
print(f"Mean IoU  : {np.mean(iou_scores):.4f}")

print("=" * 50)

**Download checkpoint về máy**

In [ ]:
# =========================================================
# DOWNLOAD CHECKPOINT _ LOG
# =========================================================

!zip -r results.zip checkpoints logs
from google.colab import files
files.download("results.zip")